In [4]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd()

# Notebook notebooks/ folder 
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "raw"
DOCS_DIR = PROJECT_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
customers = pd.read_csv(DATA_DIR / "customers.csv")
merchants = pd.read_csv(DATA_DIR / "merchants.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")

# Convert amount to numeric
transactions["amount"] = pd.to_numeric(
    transactions["amount"], errors="coerce"
)

print("Customers:", customers.shape)
print("Merchants:", merchants.shape)
print("Transactions:", transactions.shape)


Customers: (10000, 8)
Merchants: (10000, 7)
Transactions: (10000, 12)


In [ ]:

# Task1. CUSTOMER ANALYSIS

# Remove extra spaces and standardize transaction status
transactions["transaction_status"] = (
    transactions["transaction_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Calculate customer transaction metrics
customer_summary = transactions.groupby(
    "customer_id", dropna=False
).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean"),
    successful_transactions=(
        "transaction_status",
        lambda x: (x == "success").sum()
    ),
    failed_transactions=(
        "transaction_status",
        lambda x: (x == "failed").sum()
    )
).reset_index()

# Sort customers by total transaction value
top_10_customers = customer_summary.sort_values(
    "total_amount", ascending=False
).head(10)

print("Customer Summary:")
display(customer_summary.head(10))

print("Top 10 Customers by Transaction Value:")
display(top_10_customers)


Customer Summary:


,customer_id,transaction_count,total_amount,average_amount,successful_transactions,failed_transactions
0,CUST00001,2,29004.03,14502.015000,2,0
1,CUST00003,2,85949.34,42974.670000,2,0
2,CUST00006,1,245.09,245.090000,1,0
3,CUST00008,3,50621.11,16873.703333,2,1
4,CUST00009,1,10658.06,10658.060000,1,0
5,CUST00010,1,685.97,685.970000,1,0
6,CUST00011,5,9926.66,1985.332000,5,0
7,CUST00012,1,1879.76,1879.760000,1,0
8,CUST00013,1,6325.99,6325.990000,0,1
9,CUST00014,1,44532.25,44532.250000,1,0


Top 10 Customers by Transaction Value:


,customer_id,transaction_count,total_amount,average_amount,successful_transactions,failed_transactions
6221,NaN,195,4222066.30,21651.622051,160,25
6220,UNKNOWN,49,980937.80,20436.204167,41,7
2211,CUST03573,5,300432.34,60086.468000,4,1
814,CUST01339,2,280373.29,93457.763333,1,1
289,CUST00479,5,270759.53,54151.906000,4,1
4710,CUST07626,4,264995.84,66248.960000,3,1
2114,CUST03423,3,253846.42,84615.473333,3,0
5690,CUST09148,5,253019.23,50603.846000,4,0
2198,CUST03551,3,244851.76,81617.253333,3,0
5197,CUST08395,4,240989.79,60247.447500,3,1


In [6]:

# 2. MERCHANT ANALYSIS

# Calculate merchant transaction metrics
merchant_summary = transactions.groupby(
    "merchant_id", dropna=False
).agg(
    transaction_count=("transaction_id", "count"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean"),
    successful_transactions=(
        "transaction_status",
        lambda x: (x == "success").sum()
    ),
    failed_transactions=(
        "transaction_status",
        lambda x: (x == "failed").sum()
    )
).reset_index()

# Add merchant details
merchant_details = merchants.drop_duplicates(
    subset=["merchant_id"]
)

merchant_summary = merchant_summary.merge(
    merchant_details[
        ["merchant_id", "merchant_name", "category",
         "merchant_type", "city", "state"]
    ],
    on="merchant_id",
    how="left"
)

# Top 10 merchants by transaction value
top_10_merchants = merchant_summary.sort_values(
    "total_amount", ascending=False
).head(10)

print("Merchant Summary:")
display(merchant_summary.head(10))

print("Top 10 Merchants by Transaction Value:")
display(
    top_10_merchants[
        ["merchant_id", "merchant_name", "category",
         "transaction_count", "total_amount",
         "successful_transactions", "failed_transactions"]
    ]
)




Merchant Summary:


,merchant_id,transaction_count,total_amount,average_amount,successful_transactions,failed_transactions,merchant_name,category,merchant_type,city,state
0,MER00002,1,64336.19,64336.190,1,0,Reliance Digital 2,Electronics,Online,Kolkata,West Bengal
1,MER00003,1,1908.31,1908.310,1,0,Apollo Pharmacy 3,Healthcare,Online,Noida,Uttar Pradesh
2,MER00004,1,69653.59,69653.590,1,0,BookMyShow 4,Entertainment,Offline,Surat,Gujarat
3,MER00005,1,524.12,524.120,1,0,Flipkart 5,Travel,Online,Pune,maharashtra
4,MER00007,2,74996.91,37498.455,2,0,MakeMyTrip 7,Fashion,Online,Kolkata,West Bengal
5,MER00008,2,31331.27,15665.635,1,1,BigBasket 8,Education,NaN,Noida,Uttar Pradesh
6,MER00009,1,1718.90,1718.900,1,0,Paytm 9,Utilities,Offline,Surat,Gujarat
7,MER00010,1,1236.32,1236.320,1,0,Amazon India 10,Automotive,Online,Pune,Maharashtra
8,MER00013,1,28451.05,28451.050,1,0,Apollo Pharmacy 13,Food Delivery,Online,Noida,Uttar Pradesh
9,MER00014,1,6296.85,6296.850,1,0,BookMyShow 14,Electronics,Offline,Surat,Gujarat


Top 10 Merchants by Transaction Value:


,merchant_id,merchant_name,category,transaction_count,total_amount,successful_transactions,failed_transactions
6176,NaN,BookMyShow 124,NaN,194,4426685.68,168,14
6175,UNKNOWN,NaN,NaN,49,1692836.18,39,3
5153,MER08313,Apollo Pharmacy 8313,Utilities,5,277429.56,5,0
1403,MER02235,Flipkart 2235,Healthcare,3,266814.62,3,0
4403,MER07075,Flipkart 7075,Fashion,8,251766.39,4,1
2711,MER04374,BookMyShow 4374,Grocery,4,244706.71,2,2
4846,MER07813,Apollo Pharmacy 7813,Food Delivery,4,239407.61,3,1
5876,MER09506,Myntra 9506,Electronics,3,222020.31,3,0
3625,MER05846,Myntra 5846,Electronics,3,217153.30,3,0
5235,MER08456,Myntra 8456,Education,3,214014.03,2,1


In [7]:
#Task 3
#check missing value
print("customer data-missing value")
display(customers.isnull().sum().to_frame("missing_values"))

print("merchant data-missing values")
display(merchants.isnull().sum().to_frame("missing_values"))

print("Transaction data-missing values")
display(transactions.isnull().sum().to_frame("missing_values"))

#check duplicate rows

print("\n duplicate rows")
print("customers",customers.duplicated().sum())
print("merchants",merchants.duplicated().sum())
print("Transaction",transactions.duplicated().sum())

#check missing IDs
print("\nmissing Ids")
print("missing customer IDs:",transactions["customer_id"].isna().sum())
print("missing merchant IDs:",transactions["merchant_id"].isna().sum())

#check invaild amount 
transactions["amount"]=pd.to_numeric(
    transactions["amount"],errors="coerce"

)
invaild_amounts=transactions[
    transactions["amount"].isna()|(transactions["amount"]<0)
]
print("\nInvaild transaction amounts:",len(invaild_amounts))
display(invaild_amounts.head(10))

customer data-missing value


,missing_values
customer_id,152
customer_name,147
email,148
phone,148
city,149
state,147
customer_type,148
registration_date,149


merchant data-missing values


,missing_values
merchant_id,153
merchant_name,154
category,149
city,149
state,149
merchant_type,149
registration_date,147


Transaction data-missing values


,missing_values
transaction_id,151
customer_id,200
merchant_id,197
payment_method_id,197
transaction_date,151
amount,202
currency,149
transaction_status,149
transaction_type,149
channel,150



 duplicate rows
customers 117
merchants 119
Transaction 118

missing Ids
missing customer IDs: 200
missing merchant IDs: 197

Invaild transaction amounts: 251


,transaction_id,customer_id,merchant_id,payment_method_id,transaction_date,amount,currency,transaction_status,transaction_type,channel,city,state
22,TXN000023,CUST08371,MER04561,PM09379,2024-05-07 11:06:05,NaN,INR,success,Purchase,POS,Mumbai,Maharashtra
24,TXN000025,CUST02269,MER01516,PM05172,2024-06-02 13:06:56,NaN,INR,success,Purchase,Mobile,Gurugram,Haryana
68,TXN000069,CUST05651,MER03195,PM04993,2024-02-06 17:57:47,-1282.41,INR,success,Purchase,Web,Nagpur,Maharashtra
108,TXN000109,CUST05611,MER03131,PM05682,NaN,NaN,INR,success,Payment,Mobile,Nagpur,Maharashtra
122,TXN000123,CUST07714,MER01203,PM04493,2024-02-13 07:42:10,NaN,INR,success,Purchase,NaN,Bengaluru,Karnataka
155,TXN000156,CUST04975,MER07873,PM02646,2024-04-19 08:21:21,NaN,INR,success,Payment,Mobile,Gurugram,Haryana
210,TXN000211,CUST06427,MER05914,PM05342,2024-09-03 04:13:03,NaN,INR,success,Purchase,Mobile,Chennai,Tamil Nadu
226,TXN000227,CUST05505,MER07910,PM00890,2024-06-19 03:54:53,-8016.90,INR,success,Purchase,Web,Chennai,NaN
257,TXN000258,CUST07897,MER07416,PM08489,2024-05-25 03:49:29,NaN,INR,success,Purchase,Mobile,Kochi,Kerala
294,TXN000295,CUST02181,MER04289,PM07602,2024-08-22 01:11:04,NaN,INR,success,Purchase,Mobile,Hyderabad,Telangana


In [ ]:
#visualization

transactions["amount"]=pd.to_numeric
top_10_customers 